In [49]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

load_dotenv()
project_root = os.getenv("PROJECT_ROOT")

In [19]:
data_dict = pd.read_csv(f"{project_root}/data/data_dictionary.csv")
df = pd.read_csv(f"{project_root}/data/Train.csv")

data_dict.info()

<class 'pandas.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   column_name    184 non-null    str  
 1   data_type      184 non-null    str  
 2   feature_group  184 non-null    str  
 3   description    184 non-null    str  
dtypes: str(4)
memory usage: 5.9 KB


In [24]:
data_dict

,column_name,data_type,feature_group,description
0,ID,string,identifier,Unique row identifier for each customer snapsh...
1,arpu,float64,customer_profile,Average revenue per user for the customer over...
2,age,int64,customer_profile,Customer age in years.
3,gender,object,customer_profile,Customer gender.
4,region,object,customer_profile,Customer region or geographic area.
...,...,...,...,...
179,m3_daily_avg_bal,float64,balance,Average daily mobile money balance in month M3...
180,m4_daily_avg_bal,float64,balance,Average daily mobile money balance in month M4...
181,m5_daily_avg_bal,float64,balance,Average daily mobile money balance in month M5...
182,m6_daily_avg_bal,float64,balance,Average daily mobile money balance in month M6...


Features are classified into several feature groups.
 - How many? 6
 - Which features matter?

STEP 1. FEATURE SELECTION =================================================

In [20]:
df

,arpu,age,gender,region,smartphone,segment,earning_pattern,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,...,m6_withdraw_highest_amount,m6_withdraw_agents,m1_daily_avg_bal,m2_daily_avg_bal,m3_daily_avg_bal,m4_daily_avg_bal,m5_daily_avg_bal,m6_daily_avg_bal,liquidity_stress_next_30d,ID
0,2653.88,47,F,Central Zone,Yes,MVC,Monthly Earner,0.6905,8,58372.55,...,0.00,0.0,12488.72,11404.86,12943.70,14157.61,13202.11,4336.98,1,ID_C1DF84C4D2
1,2653.88,47,F,Central Zone,Yes,MVC,Monthly Earner,0.6905,0,0.00,...,5191.14,4.0,13677.61,2781.25,5897.59,5416.23,11062.23,4849.68,0,ID_CB8B764D76
2,2653.88,47,F,Central Zone,Yes,MVC,Monthly Earner,0.6905,12,41262.86,...,0.00,0.0,7271.09,5364.41,9023.13,13001.02,17107.92,15174.32,1,ID_D40D58FC67
3,2653.88,47,F,Central Zone,Yes,MVC,Monthly Earner,0.6905,10,8738.58,...,8965.63,3.0,14331.10,33465.26,2973.45,3577.07,18990.91,8488.26,0,ID_CFD8F26581
4,20535.82,63,F,Northern Highlands,No,MVC,Weekly Earner,0.8959,2,3654.68,...,0.00,0.0,5602.75,8497.49,26252.75,7618.93,9290.65,4214.71,0,ID_20EB501231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,1554.59,30,F,Southern Highlands,No,LVC,Weekly Earner,0.1637,0,0.00,...,0.00,0.0,1605.52,2182.85,720.60,887.05,803.77,2613.55,0,ID_D8862FBF78
39996,26548.53,57,M,Islander,No,MVC,Weekly Earner,0.7843,0,0.00,...,14442.30,2.0,11480.26,5929.57,9268.63,6221.86,7364.33,7370.45,0,ID_0DE7DD593F
39997,26548.53,57,M,Islander,No,MVC,Weekly Earner,0.7843,8,17528.25,...,4451.08,2.0,8700.32,10870.39,10766.04,11739.36,12240.99,5011.35,0,ID_978629D112
39998,26548.53,57,M,Islander,No,MVC,Weekly Earner,0.7843,0,0.00,...,6183.36,1.0,5676.57,2523.80,13305.51,8698.72,2442.02,11312.53,0,ID_91ECE231B0


In [41]:
feature_groups = data_dict["feature_group"]

for i in feature_groups.unique():
    print(i)

identifier
customer_profile
activity
behaviour
balance
target


In [26]:
df.describe()

,arpu,age,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_volume,m2_paybill_total_value,m2_paybill_highest_amount,...,m6_withdraw_total_value,m6_withdraw_highest_amount,m6_withdraw_agents,m1_daily_avg_bal,m2_daily_avg_bal,m3_daily_avg_bal,m4_daily_avg_bal,m5_daily_avg_bal,m6_daily_avg_bal,liquidity_stress_next_30d
count,40000.000000,40000.00000,40000.000000,40000.000000,4.000000e+04,40000.000000,40000.000000,40000.000000,4.000000e+04,40000.000000,...,4.000000e+04,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000
mean,24480.982653,44.20050,0.504210,3.178500,3.355774e+04,3463.667285,2.127475,3.215975,3.366216e+04,3520.387921,...,1.028221e+05,9297.774969,1.977875,14684.315477,14610.007011,14634.600378,14711.700646,14809.994068,14684.074900,0.150000
std,51992.965165,15.32212,0.229379,4.964303,1.246310e+05,8290.539405,2.743577,5.029469,1.268042e+05,8465.908039,...,3.555190e+05,19146.708926,1.832040,32739.081130,32250.370737,32287.777394,32675.269386,33056.285393,32921.604867,0.357076
min,100.920000,18.00000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000,...,0.000000e+00,0.000000,0.000000,139.520000,137.820000,158.150000,97.330000,139.520000,163.040000,0.000000
25%,2809.712500,31.00000,0.332675,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000,...,0.000000e+00,0.000000,0.000000,1445.915000,1441.682500,1438.710000,1454.952500,1438.920000,1439.532500,0.000000
50%,7070.795000,44.00000,0.508450,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000,...,6.141555e+03,2025.410000,2.000000,5082.590000,5076.255000,5080.305000,5066.015000,5072.920000,5060.740000,0.000000
75%,21021.690000,57.00000,0.667325,5.000000,1.270061e+04,3431.962500,4.000000,5.000000,1.282144e+04,3436.347500,...,4.930610e+04,10361.752500,4.000000,11544.090000,11478.565000,11502.472500,11508.355000,11518.010000,11456.060000,0.000000
max,299815.190000,70.00000,1.000000,41.000000,2.987089e+06,49998.420000,8.000000,38.000000,4.202311e+06,49997.310000,...,8.153060e+06,112480.440000,5.000000,529537.290000,667242.510000,516521.790000,539574.450000,499853.500000,486015.760000,1.000000


In [50]:
x = df.drop(["liquidity_stress_next_30d"], axis=1)
y = df["liquidity_stress_next_30d"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=1)

# 1. Deal with missing values - numerical-> fill with the median, categorical -> fill with the most frequent
# identify numerical and categorical columns
num_cols = x_train.select_dtypes(include=['int64', 'float']).columns
cat_cols = x_train.select_dtypes(include=['object', 'category']).columns

# impute numerical data
num_imputer = SimpleImputer(strategy="median")
x_train[num_cols] = num_imputer.fit_transform(x_train[num_cols])
x_test[num_cols] = num_imputer.transform(x_test[num_cols])

# impute categorical data
cat_imputer = SimpleImputer(strategy="most_frequent", fill_value="missing")
x_train = cat_imputer.fit_transform(x_train[cat_cols])
x_test = cat_imputer.transform(x_test[cat_cols])

print(x_train)

/tmp/ipykernel_2429440/1635935044.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = x_train.select_dtypes(include=['object', 'category']).columns


[['F' 'Coastal & Eastern Corridor' 'No' 'MVC' 'Weekly Earner'
  'ID_8D8AB92EF2']
 ['F' 'Islander' 'No' 'MVC' 'Weekly Earner' 'ID_792673676B']
 ['M' 'Western Zone' 'Yes' 'MVC' 'Weekly Earner' 'ID_A6496EE224']
 ...
 ['F' 'Southern Highlands' 'Yes' 'MVC' 'Irregular Earner' 'ID_0148B05ADF']
 ['F' 'Islander' 'Yes' 'MVC' 'Monthly Earner' 'ID_FEC9A4F864']
 ['M' 'Islander' 'Yes' 'LVC' 'Irregular Earner' 'ID_47BBAF6345']]
